# Notebook 02 — Graph-Reasoning Training Dataset Construction

**Paper:** *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design*

This notebook generates the **Graph-PRefLexOR training dataset** (`data/training/graph_reasoning_v3.jsonl`) used to fine-tune Graph-PRefLexOR models via GRPO. Each example contains a scientific question drawn from a research paper corpus, paired with:

- **`chosen`** — a structured reasoning trace in the Graph-PRefLexOR format:
  `<think> <brainstorm> … <graph> … <graph_json> … <patterns> … <synthesis> </think>` + final answer
- **`rejected`** — a shallow, direct answer with no reasoning structure
- **`teacher_graph_json`** — the validated knowledge graph JSON

**Model used:** `gpt-5.4` (teacher for structured answers), `gpt-5.4` (rejected answers)  
**Corpus:** `data/corpus/consolidated_papers.json`  
**Output:** `data/training/graph_reasoning_v3.jsonl`

**Prerequisites:** `pip install openai datasets python-dotenv tqdm pydantic`

## 1. Imports and Sentinel Definitions

The Graph-PRefLexOR format uses XML-style sentinels to delimit reasoning phases. These are the same tokens the model learns to produce during GRPO training.

In [ ]:
import argparse
import json
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from datasets import Dataset, load_dataset
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

# Load API key from repo root .env
load_dotenv(Path("../.env"))
openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY not set. Add it to .env at the repo root.")

# ── Reasoning sentinel tokens (must match the GRPO training config) ──────────
THINK_START      = "<think>"
THINK_END        = "</think>"
BRAINSTORM_START = "<brainstorm>"
BRAINSTORM_END   = "</brainstorm>"
GRAPH_START      = "<graph>"
GRAPH_END        = "</graph>"
GRAPH_JSON_START = "<graph_json>"
GRAPH_JSON_END   = "</graph_json>"
PATTERNS_START   = "<patterns>"
PATTERNS_END     = "</patterns>"
SYNTHESIS_START  = "<synthesis>"
SYNTHESIS_END    = "</synthesis>"

# Maximum context length passed to the teacher model (characters, ~8k tokens)
TRUNC_LEN = 32_000


## 2. Graph JSON Schema

The knowledge graph produced in `<graph_json>` must conform to this schema:
nodes have a simple `id` string; edges have `source`, `relation`, `target`.
The `relation` values are drawn from a predefined relational vocabulary (e.g., `encodes`, `enables`, `causes`, `inhibits`, `modulates`).

In [ ]:
class Node(BaseModel):
    id: str = Field(..., description="Short CamelCase identifier, no spaces")

class Edge(BaseModel):
    source:   str
    relation: Optional[str] = None
    target:   str

class GraphJSON(BaseModel):
    """Validated schema for the knowledge graph embedded in <graph_json>."""
    nodes: List[Node]
    edges: List[Edge] = Field(default_factory=list)


## 3. LLM Client and Helper Utilities

In [ ]:
def get_llm_client(api_key: str, base_url: Optional[str] = None,
                   timeout: float = 120.0) -> OpenAI:
    """Create an OpenAI client, optionally pointing at a custom base URL."""
    params: Dict[str, Any] = {"api_key": api_key}
    if base_url:
        params["base_url"] = base_url
    if timeout:
        params["timeout"] = timeout
    return OpenAI(**params)


def llm_call(client: OpenAI, model: str, system_prompt: str, user_prompt: str) -> Optional[str]:
    """Call the model via Responses API, falling back to Chat Completions."""
    # Try Responses API (gpt-5.x)
    try:
        resp = client.responses.create(
            model=model,
            input=[
                {"role": "system", "content": [{"type": "input_text", "text": system_prompt}]},
                {"role": "user",   "content": [{"type": "input_text", "text": user_prompt}]},
            ],
        )
        return (resp.output_text or "").strip()
    except Exception:
        pass

    # Fallback to Chat Completions API (gpt-4o and earlier)
    try:
        chat = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
        return (chat.choices[0].message.content or "").strip()
    except Exception as e:
        print(f"[llm_call] Both APIs failed: {e}")
        return None


## 4. Teacher: Structured Answer Generation

The teacher model produces the `chosen` completion — a full Graph-PRefLexOR reasoning trace followed by a comprehensive final answer. This is the format the GRPO-trained model learns to emulate.

In [ ]:
def teacher_generate_structured_answer(client: OpenAI, model: str,
                                        question: str, context: str) -> Optional[str]:
    """Generate a structured Graph-PRefLexOR reasoning trace.

    The output structure mirrors what the fine-tuned model must produce:
      <think>
        <brainstorm> … divergent exploration … </brainstorm>
        <graph>      … text-form entity/relation map … </graph>
        <graph_json> … validated JSON graph … </graph_json>
        <patterns>   … higher-order causal patterns … </patterns>
        <synthesis>  … integrated hypothesis … </synthesis>
      </think>
      [final answer paragraph(s)]
    """
    sys = "You are a materials and mechanistic reasoning expert. Always reason via a graph-based latent structure."
    user = f"""
Answer the following question in TWO phases.

PHASE 1 — Structured reasoning trace (EXACTLY this template):

{THINK_START}
{BRAINSTORM_START}
Freely explore the problem space. What concepts, mechanisms, and phenomena are relevant?
{BRAINSTORM_END}

{GRAPH_START}
Identify core entities and describe directed relationships in plain text.
{GRAPH_END}

{GRAPH_JSON_START}
Provide a STRICT JSON with ONLY "nodes" and "edges":
{{"nodes": [{{"id": "A"}}, {{"id": "B"}}], "edges": [{{"source": "A", "relation": "influences", "target": "B"}}]}}
{GRAPH_JSON_END}

{PATTERNS_START}
Describe 1–2 abstract patterns or laws from the graph (use →, ↑, ↓, ∝ where helpful).
{PATTERNS_END}

{SYNTHESIS_START}
Integrate brainstorm, graph, and patterns into a coherent, testable hypothesis.
{SYNTHESIS_END}
{THINK_END}

PHASE 2 — Write a thorough, expert-level final answer (multiple paragraphs).

Question: {question}
Context:  {context}
"""
    return llm_call(client, model, sys, user)


def teacher_generate_rejected(client: OpenAI, model: str,
                               question: str, context: str) -> Optional[str]:
    """Generate a shallow, direct answer with no reasoning trace (the 'rejected' response)."""
    sys  = "You are a hurried scientist who gives very short, shallow answers."
    user = f"""Answer in 1–3 sentences WITHOUT showing reasoning or using special tokens.
Question: {question}
Context:  {context}"""
    return llm_call(client, model, sys, user)


## 5. Parsing Helpers

Extract and validate individual blocks from the model's raw output.

In [ ]:
def find_tag_block(s: str, start_tag: str, end_tag: str) -> Optional[str]:
    """Return the text between start_tag and end_tag, or None if not found."""
    i = s.find(start_tag)
    if i == -1:
        return None
    j = s.find(end_tag, i + len(start_tag))
    if j == -1:
        return None
    return s[i + len(start_tag): j].strip()


def extract_graph_json_block(chosen: str) -> Optional[str]:
    """Extract and validate the <graph_json> block from a chosen completion."""
    inner = find_tag_block(chosen, GRAPH_JSON_START, GRAPH_JSON_END)
    if not inner:
        return None
    try:
        obj = json.loads(inner)
        _ = GraphJSON(**obj)          # validate against schema
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return None  # skip malformed graphs


def extract_final_answer(chosen: str) -> str:
    """Return everything after </think> as the final answer."""
    idx = chosen.find(THINK_END)
    if idx == -1:
        return chosen.strip()
    return chosen[idx + len(THINK_END):].strip()


## 6. Dataset Builder

Iterates over the paper corpus, generates (question, chosen, rejected) triples, and saves them incrementally to disk.

In [ ]:
def load_local_corpus(consolidated_json_path: str) -> Dataset:
    """Load the consolidated paper corpus (JSON or JSONL) as a HuggingFace Dataset."""
    p = Path(consolidated_json_path)
    raw = p.read_text(encoding="utf-8", errors="replace").strip()

    records: List[Dict[str, Any]] = []
    try:
        if raw[0] == "[":
            records = [r for r in json.loads(raw) if isinstance(r, dict)]
        else:
            for line in raw.splitlines():
                line = line.strip()
                if line:
                    try:
                        r = json.loads(line)
                        if isinstance(r, dict):
                            records.append(r)
                    except Exception:
                        pass
    except Exception:
        pass

    # Merge section fields into a single "text" key if absent
    rows = []
    for rec in records:
        text = rec.get("text") or "\n\n".join(
            f"## {s.upper()}\n{rec[s]}" for s in ["abstract","results","discussion","conclusion"]
            if rec.get(s)
        )
        if len(text) >= 200:
            rows.append({"text": text, "title": rec.get("title"), "doi": rec.get("doi")})

    return Dataset.from_list(rows)


def build_graph_reasoning_dataset(
    corpus_path:   str,
    teacher_model: str,
    num_examples:  int,
    output_path:   str,
    save_steps:    int = 10,
    resume:        bool = True,
) -> Dataset:
    """Build the Graph-PRefLexOR training dataset from a local paper corpus.

    Args:
        corpus_path:   Path to consolidated_papers.json.
        teacher_model: Model ID for both chosen and rejected generation.
        num_examples:  Target number of training examples to generate.
        output_path:   Where to save the JSONL dataset.
        save_steps:    Checkpoint every N examples.
        resume:        If True, skip examples already saved in output_path.
    """
    teacher_client = get_llm_client(api_key=openai_key)
    corpus = load_local_corpus(corpus_path)
    print(f"Corpus size: {len(corpus)} papers")

    # Resume from existing output
    rows: List[Dict[str, Any]] = []
    if resume and os.path.exists(output_path):
        existing = load_dataset("json", data_files=output_path, split="train")
        rows = [dict(r) for r in existing]
        print(f"Resuming from {output_path}: {len(rows)} examples already generated")

    if len(rows) >= num_examples:
        print(f"Already have {len(rows)} examples. Nothing to do.")
        return Dataset.from_list(rows)

    indices = list(range(len(corpus)))
    random.shuffle(indices)
    last_save = len(rows)
    skipped   = 0

    pbar = tqdm(total=num_examples, initial=len(rows), desc="Building dataset")

    for idx in indices:
        if len(rows) >= num_examples:
            break

        row     = corpus[idx]
        context = row.get("text", "")
        if len(context) < 200:
            skipped += 1
            continue
        context = context[:TRUNC_LEN]

        # Generate question from context
        q = llm_call(teacher_client, teacher_model,
                     "You are a scientist who designs deep, standalone questions.",
                     f"Write one challenging scientific question from:\n\n{context}\n\nQuestion:")
        if not q:
            skipped += 1
            continue
        q = q.split("\n")[0].strip()
        if not q.endswith("?"):
            q += "?"

        # Generate structured (chosen) and shallow (rejected) answers
        chosen  = teacher_generate_structured_answer(teacher_client, teacher_model, q, context)
        if not chosen:
            skipped += 1
            continue

        graph_json_str = extract_graph_json_block(chosen)
        final_answer   = extract_final_answer(chosen)
        if not graph_json_str or not final_answer:
            skipped += 1
            continue

        rejected = teacher_generate_rejected(teacher_client, teacher_model, q, context)
        if not rejected:
            skipped += 1
            continue

        rows.append({
            "prompt":             q.strip(),
            "answer":             final_answer.strip(),
            "chosen":             chosen.strip(),
            "rejected":           rejected.strip(),
            "teacher_graph_json": graph_json_str,
        })
        pbar.update(1)
        pbar.set_postfix({"skipped": skipped})

        # Periodic checkpoint save
        if len(rows) - last_save >= save_steps:
            last_save = len(rows)
            Dataset.from_list(rows).to_json(output_path, lines=True)
            tqdm.write(f"[checkpoint] {len(rows)} examples → {output_path}")

    pbar.close()

    # Final save
    ds = Dataset.from_list(rows)
    ds.to_json(output_path, lines=True)
    print(f"Done. {len(rows)} examples saved to {output_path} (skipped {skipped})")
    return ds


## 7. Run Dataset Generation

Generates up to 60 training examples from the paper corpus. The output is saved to `data/training/graph_reasoning_v3.jsonl`. Adjust `num_examples` as needed (the paper uses the full corpus).

In [ ]:
CORPUS_PATH  = "../data/corpus/consolidated_papers.json"
OUTPUT_PATH  = "../data/training/graph_reasoning_v3.jsonl"
TEACHER_MODEL = "gpt-5.4"

# Build dataset — comment out after first run to avoid re-generating
# ds = build_graph_reasoning_dataset(
#     corpus_path   = CORPUS_PATH,
#     teacher_model = TEACHER_MODEL,
#     num_examples  = 60,
#     output_path   = OUTPUT_PATH,
#     save_steps    = 5,
#     resume        = True,
# )


## 8. Inspect the Generated Dataset

In [ ]:
import pandas as pd

ds_df = pd.read_json(OUTPUT_PATH, lines=True)
print(f"Dataset size: {len(ds_df)} examples")
print("\nColumns:", ds_df.columns.tolist())
print("\nSample prompt:")
print(ds_df['prompt'].iloc[0])
print("\nGraph node count (first example):")
gj = json.loads(ds_df['teacher_graph_json'].iloc[0])
print(f"  {len(gj['nodes'])} nodes, {len(gj['edges'])} edges")

ds_df.to_csv(OUTPUT_PATH.replace(".jsonl", ".csv"), index=False)
print("\nCSV also saved.")
